In [ ]:
import time
import requests
import cv2
import numpy as np
import ipywidgets as widgets
import threading
from IPython.display import display
from pop import Pilot

# 1. 설정
URL = "https://detect.roboflow.com/autocar-q0bqc/1"
PARAMS = {"api_key": "5qqbBd2raZ4VbubgCOgG", "confidence": "40", "format": "json"}

# 카메라 초기화 - 오류 방지를 위해 설정 단순화
try:
    cam = Pilot.Camera(width=320, height=320)
except:
    print("카메라 초기화 실패. 재시작하거나 권한을 확인하세요.")

ac = Pilot.AutoCar()
image_widget = widgets.Image(format='jpeg', width=320, height=320)
display(image_widget)

latest_frame = np.zeros((320, 320, 3), dtype=np.uint8)
latest_preds = []

# 2. AI 전담 스레드
def ai_worker():
    global latest_preds, latest_frame
    while True:
        # 최신 프레임을 안전하게 복사해서 사용
        frame_to_send = latest_frame.copy()
        
        if frame_to_send is not None and frame_to_send.shape[0] == 320:
            _, img_encoded = cv2.imencode('.jpg', frame_to_send)
            try:
                res = requests.post(URL, params=PARAMS, files={"file": img_encoded.tobytes()}, timeout=0.3)
                latest_preds = res.json().get('predictions', [])
            except: pass
        time.sleep(0.05)

threading.Thread(target=ai_worker, daemon=True).start()

print("🚀 [화면 출력 최적화] 오토카 추종 모드 시작!")
try:
    while True:
        # 1. 카메라 데이터 획득 - 가장 확실한 방식인 .value 사용
        # 만약 여기서 에러가 난다면 Pilot 라이브러리 버전 문제입니다.
        camera_data = cam.value
        
        if camera_data is None:
            continue
            
        # 2. 데이터 포맷 통일 (카메라 포맷에 따라 bytes일 수도 있음)
        if isinstance(camera_data, bytes):
            nparr = np.frombuffer(camera_data, dtype=np.uint8)
            frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        else:
            # 이미 numpy 배열 형태인 경우
            frame = camera_data
            
        # 3. 화면 표시용 프레임 업데이트 (AI 스레드가 가져갈 수 있도록)
        if frame is not None and frame.shape == (320, 320, 3):
            latest_frame = frame.copy()
            
            # 주행 로직
            if latest_preds:
                target = latest_preds[0]
                x, y = int(target['x']), int(target['y'])
                w, h = int(target['width']), int(target['height'])
                
                # 박스 그리기
                cv2.rectangle(frame, (x - w//2, y - h//2), (x + w//2, y + h//2), (0, 255, 0), 2)
                cv2.circle(frame, (x, y), 5, (0, 0, 255), -1) # 중심점

                # 주행 제어
                ac.steering = np.clip((x - 160) / 160 * 3.0, -1.0, 1.0)
                size_rate = (w * h) / (320 * 320)
                if 0.25 < size_rate < 0.35: ac.stop()
                elif size_rate <= 0.25: ac.forward(40)
                else: ac.backward(40)
            else:
                ac.stop()

            # 4. [핵심] 위젯에 강제로 JPEG 바이트 밀어넣기
            _, jpeg = cv2.imencode('.jpg', frame)
            image_widget.value = jpeg.tobytes()
            
        # 루프 속도 조절 (너무 빨라서 과부하 걸리는 것 방지)
        time.sleep(0.01)

except KeyboardInterrupt:
    print("\n✨ 종료 중...")
    ac.stop()
    # cam.stop() # 일부 버전에서는 stop()이 없을 수 있음